In [0]:
%run ./01-config

In [0]:
# Import PySpark SQL functions for transformations (e.g., current_timestamp, input_file_name, to_date)
from pyspark.sql import functions as F

# Construct absolute paths for raw data ingestion and streaming checkpoints
# - `landing_zone`: Base directory where raw source files (CSV/JSON) are staged for ingestion
# - `checkpoint_base`: Directory to store streaming query offsets and state (required for fault tolerance)
landing_zone = base_dir_data + "/raw"
checkpoint_base = base_dir_checkpoint + "/checkpoints"

# Set the active Spark database context to ensure all subsequent table operations target the correct namespace
spark.sql(f"USE {catalog}.{db_name}")


# --------------------------------------------------------------------------------------------------
# BRONZE LAYER STREAMING INGESTION FUNCTIONS
# Each function consumes raw files from cloud storage and writes them as Delta tables in append-only mode.
# --------------------------------------------------------------------------------------------------

def consume_user_registration(once = True, processing_time = "5 seconds"):
    """
    Ingests user registration events from CSV files in the landing zone into the Bronze table 
    `registered_users_bz`. Adds metadata columns (`load_time`, `source_file`) for auditability.
    
    Parameters:
      once (bool): If True, runs as a batch (trigger once); if False, runs as a continuous stream.
      processing_time (str): Interval for micro-batch triggers (e.g., "5 seconds") when `once=False`.
    """
    # Define expected schema of raw CSV files (matches source system format)
    schema = "user_id long, device_id long, mac_address string, registration_timestamp double"

    # Configure Auto Loader (cloudFiles) to read CSV files from the registered_users_bz subdirectory
    df_stream = (spark.readStream
                    .format("cloudfiles")                     # Use Databricks Auto Loader
                    .schema(schema)                           # Explicit schema avoids inference
                    .option("cloudFiles.maxFilesPerTrigger", 1)  # Process 1 file per micro-batch
                    .option("cloudFiles.format", "csv")       # Source file format
                    .option("header", True)                   # First row contains column names
                    .load(landing_zone + "/registered_users_bz")  # Source path
                    .withColumn("load_time", F.current_timestamp())     # Add ingestion timestamp
                    .withColumn("source_file", F.input_file_name())    # Track originating file
                )

    # Configure streaming write to Delta table in append mode (Bronze = immutable raw data)
    stream_writer = df_stream.writeStream \
        .format("delta") \
        .option("checkpointLocation", checkpoint_base + "/registered_users_bz") \
        .outputMode("append")\
        .queryName("dev.sbit_db.registered_users_bz")  # Unique name for monitoring in Spark UI

    # Trigger execution: either as a one-time batch or continuous stream
    if once == True:
        return stream_writer.trigger(availableNow=True).toTable(f"{catalog}.{db_name}.registered_users_bz")
    else:
        return stream_writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{db_name}.registered_users_bz")


def consume_gym_logins(once = True, processing_time = "5 seconds"):
    """
    Ingests gym login/logout events from CSV files into the Bronze table `gym_logins_bz`.
    Similar pattern to `consume_user_registration` but for a different data source.
    """
    schema = "mac_address string, gym bigint, login double, logout double"

    df_stream = (spark.readStream 
                    .format("cloudFiles") 
                    .schema(schema) 
                    .option("maxFilesPerTrigger", 1)          # Note: shorthand option name (still valid)
                    .option("cloudFiles.format", "csv") 
                    .option("header", "true")                 # String "true" is equivalent to boolean True
                    .load(landing_zone + "/gym_logins_bz") 
                    .withColumn("load_time", F.current_timestamp())
                    .withColumn("source_file", F.input_file_name())
                )

    stream_writer = df_stream.writeStream \
                                .format("delta") \
                                .option("checkpointLocation", checkpoint_base + "/gym_logins_bz") \
                                .outputMode("append") \
                                .queryName("gym_logins_bz_ingestion_stream")

    if once == True:
        return stream_writer.trigger(availableNow=True).toTable(f"{catalog}.{db_name}.gym_logins_bz")
    else:
        return stream_writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{db_name}.gym_logins_bz")


def consume_kafka_multiplex(once = True, processing_time = "5 seconds"):
    """
    Ingests multiplexed Kafka messages (in JSON format) into the Bronze table `kafka_multiplex_bz`.
    Enriches each record with date and `week_part` from the `date_lookup` table for partitioning.
    
    Note: Kafka timestamps are in milliseconds (hence `/1000` before casting to timestamp).
    """
    schema = "key string, value string, topic string, partition bigint, offset bigint, timestamp bigint"
    
    # Load the static date dimension table for enrichment (broadcasted for efficiency)
    df_date_lookup = spark.table(f"{catalog}.{db_name}.date_lookup").select("date", "week_part")

    df_stream = (spark.readStream
                    .format("cloudFiles")
                    .schema(schema)
                    .option("maxFilesPerTrigger", 5)          # Process up to 5 files per micro-batch
                    .option("cloudFiles.format", "json")      # Source format is JSON
                    .load(landing_zone + "/kafka_multiplex_bz")                        
                    .withColumn("load_time", F.current_timestamp())       
                    .withColumn("source_file", F.input_file_name())
                    # Join with date_lookup to add `date` and `week_part` for partitioning
                    .join(F.broadcast(df_date_lookup), 
                            [F.to_date((F.col("timestamp")/1000).cast("timestamp")) == F.col("date")], 
                            "left")  # Left join preserves all Kafka records even if date missing
                )

    stream_writer = df_stream.writeStream \
                                .format("delta") \
                                .option("checkpointLocation", checkpoint_base + "/kafka_multiplex_bz") \
                                .outputMode("append") \
                                .queryName("kafka_multiplex_bz_ingestion_stream")

    if once == True:
        return stream_writer.trigger(availableNow=True).toTable(f"{catalog}.{db_name}.kafka_multiplex_bz")
    else:
        return stream_writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{db_name}.kafka_multiplex_bz")
    


def consume_bronze(once=True, processing_time="5 seconds"):
    """
    Orchestrates ingestion of all Bronze layer tables.
    
    If `once=True`, waits for all streaming queries to complete (using `awaitTermination`)
    before proceeding—ensuring batch-like behavior for testing or historical loads.
    """
    import time
    start = int(time.time())
    print(f"\nStarting bronze layer consumption ...")
    consume_user_registration(once, processing_time) 
    consume_gym_logins(once, processing_time) 
    consume_kafka_multiplex(once, processing_time)
    if once:
        # Block until all triggered batch queries finish
        for stream in spark.streams.active:
            stream.awaitTermination()
    print(f"Completed bronze layer consumtion {int(time.time()) - start} seconds")


# --------------------------------------------------------------------------------------------------
# VALIDATION FUNCTIONS
# Verify that expected record counts exist in Bronze tables after ingestion.
# --------------------------------------------------------------------------------------------------

def assert_count(catalog, db_name, table_name, expected_count, filter="true"):
    """
    Validates that a table contains exactly `expected_count` records matching an optional filter.
    
    Parameters:
      filter (str): SQL boolean expression (default "true" = all rows)
    """
    print(f"Validating record counts in {table_name}...", end='')
    actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").where(filter).count()
    assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name} where {filter}" 
    print(f"Found {actual_count:,} / Expected {expected_count:,} records where {filter}: Success")        
    
def validate_bronze(catalog, db_name, sets):
    """
    Validates record counts across all Bronze tables based on the number of test data sets loaded.

    - `sets=1`: Expected counts for a single batch of test data.
    - `sets=2`: Expected counts for two batches (e.g., after re-running ingestion).

    Specific topic-based counts are validated for the multiplexed Kafka table.
    """
    import time
    start = int(time.time())
    print(f"\nValidating bronze layer records...")
    assert_count(catalog, db_name, "registered_users_bz", 5 if sets == 1 else 10)
    assert_count(catalog, db_name, "gym_logins_bz", 8 if sets == 1 else 16)
    # Validate counts by Kafka topic in the multiplexed table
    assert_count(catalog, db_name, "kafka_multiplex_bz", 7 if sets == 1 else 13, "topic='user_info'")
    assert_count(catalog, db_name, "kafka_multiplex_bz", 16 if sets == 1 else 32, "topic='workout'")
    assert_count(catalog, db_name, "kafka_multiplex_bz", sets * 253801, "topic='bpm'")  # Large volume of heart rate data
    print(f"Bronze layer validation completed in {int(time.time()) - start} seconds")